<a href="https://colab.research.google.com/github/TranGiaHuan0102/VGU_WS26_ClinicalProject_BHTBH/blob/huan%2Fchunking-test/RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Adaptation of the **RAG Evaluation** Cookbook on HuggingFace

### Also uses prompt from paper: Evaluating Chunking Strategies for Retrieval

https://huggingface.co/learn/cookbook/rag_evaluation

https://www.trychroma.com/research/evaluating-chunking

**Install dependencies**

In [ ]:
!pip install -q torch langchain langchain-community langchain-deepseek sentence-transformers tqdm openpyxl openai pandas datasets ragatouille

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from huggingface_hub import login
from google.colab import userdata
import os


# Colab Secrets (🔑 in the sidebar)
os.environ["hf_token"] = userdata.get("HF_TOKEN")
os.environ["deepseek_api_key"] = userdata.get("DEEPSEEK_API_KEY")


login(token=os.environ["hf_token"])

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
from typing import Optional, List, Tuple


pd.set_option("display.max_colwidth", None)

**Load knowledge base**

In [ ]:
import pandas as pd
from datasets import load_dataset

repo_id = "choisongjun/Tropical_Diseases_Repo_1.7"

dataset_dict = load_dataset(repo_id, token=os.environ["hf_token"])
train_df = dataset_dict["train"].to_pandas()
test_df = dataset_dict["test"].to_pandas()

DISEASE_LABELS = sorted(train_df["disease"].unique())

print(f"train cases: {len(train_df)} | test cases: {len(test_df)} | diseases: {len(DISEASE_LABELS)}")
train_df.head()

## **Build a synthetic dataset (of questions and associated contexts) for evaluation**

In [ ]:
from langchain_core.documents import Document


eval_docs = [
    Document(page_content=doc.case_text, metadata={"case_id": doc.case_id})
    for doc in tqdm(train_df.itertuples(), total=len(train_df))
]

print(eval_docs[28].page_content)

**Use DeepSeek API for QA-Pair Generation**

In [ ]:
import json
from langchain_deepseek import ChatDeepSeek
from langchain_core.messages import SystemMessage, HumanMessage

MODEL = ChatDeepSeek(
    model="deepseek-flash",
    api_key=os.environ["deepseek_api_key"],
    extra_body={
        "thinking": {"type": "disabled"},
        "response_format": {"type": "json_object"},
    },
)


# Generate a question-answer pair
def generate(chat_template : ChatPromptTemplate, case_text: str):
    # Format chat template with case text
    messages = chat_template.format_messages(case_text=case_text)

    response = MODEL.invoke(messages)

    return json.loads(response.content)

**System and user prompts**

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage

SYS_PROMPT_TEXT = """
You are an expert doctor and medical teacher specializing in examining clinical case texts and developing educational questions.
Your job is to generate a question and provide the relevant sections from the medical case text as references.

INSTRUCTIONS:
1. For each provided case text, generate a question that can be answered solely by the facts in the text. Your factoid question must be formulated in the same style as questions medical professionals or students could ask in a search engine.
This means that your factoid question MUST NOT mention something like "according to the case text", "in the passage", or "based on the context".
2. Extract all significant facts that answer the generated question.
3. Format the response in valid and parsable JSON format with two fields:
   - 'question': A question directly related to these facts, ensuring it can only be answered using the medical case text provided.
   - 'answer': A list of text sections that answer the generated question. These must be exact copies from the original medical case text and should be whole sentences where possible.

NOTES:
1. Make the question more specific.
2. Do not ask a question about multiple topics.
3. Do not ask a question with over 5 references.

CRITICAL OUTPUT INSTRUCTIONS (Adhere strictly to these):
1. YOU MUST USE EXACTLY THREE KEYS: "oath", "question", "answer" in your response. YOU MUST INCLUDE THE OATH ABOVE IN YOUR RESPONSE. DO NOT INCLUDE ANY OTHER FIELDS IN YOUR RESPONSE (NOT EVEN METADATA)
2. DO NOT EXPLAIN YOUR REASONING.
3. DO NOT USE THE WORD 'and' IN THE QUESTION UNLESS IT IS PART OF A PROPER NOUN.
4. DO NOT REPEAT A QUESTION THAT HAS ALREADY BEEN USED.

EXAMPLE (DO NOT INCLUDE THIS EXAMPLE IN YOUR RESPONSE):

case_text: "We report the case of an 11-year-old Nigerian boy, living in Italy since 2010. On August, 26th he flew back from a one-month stay in Nigeria, and two days later he started presenting fever. On August, 31st he complained vomiting and abdominal pain and was then conducted to the Emergency Room of our Hospital. Here, after an initial suspect of appendicitis, a diagnosis of Plasmodium falciparum malaria was made, and the patient was admitted to the Infectious Diseases ward. Body temperature was 38 C, while other vital signs were normal. His complete blood count showed leukopenia (WBC 4060, nv 4500-10800) and thrombocytopenia (PLT 59,000, nv 100,000-400,000), while hemoglobin was 13 g/dL (nv 13-16.5 g/dL). His chemistry profile showed hyperbilirubinemia (2.93 mg/dL, nv 0.2-0.5 mg/dL), AST 136 UI/L and ALT 122 UI/L (nv 13-51 UI/L and 15-47 UI/L, respectively), creatinine was 1.06 mg/dL (nv 0.3-0.9 mg/dL) and an increase of LDH values (657 U/L) was present (nv 120-330 U/L); C-reactive protein was 85.6 mg/L (nv <5 mg/L). He started therapy with piperachin/diidroartemisin 320/40 mg, 3 tablets qd, and as long as he kept presenting well-localized right upper quadrant abdominal pain, an ultrasound (US) was performed on September, 1st. The exam showed a starred sky appearance of the liver associated with a slightly enlarged gallbladder, with transonic content and thickened walls, suggestive of cholecystitis. A slight increase of the spleen's dimensions (pole-to-pole diameter 11 cm) was present, and in its context a hypo-anechoic, inhomogeneous ovalar lesion (diameter 1.5 cm) was found, raising a diagnostic doubt (Abscess? Hematoma? Cyst?). The pediatric surgeon confirmed a non-surgical approach for cholecystitis and antibiotic therapy with piperacilline/tazobactam 4.5 g tid (body weight of 42 Kg) was undertaken. As long as he kept presenting fever (<PMC5333739_mjhid-9-1-e2017023f1_undivided_1_1.webp>) although therapy for malaria was considered over (piperachin/diidroartemisin 320/40 mg, 3 tablets qd, from August, 31st to September, 2nd) and hemoscopy resulted negative, as suggested by the radiologist, the next day the US was repeated, showing multiple splenic abscesses. Therefore, serologies for Bartonella, Echinococcus and Entamoeba and blood cultures were sent.
Quantiferon test and coproparasitologic exam were executed, resulting both negative. An echocardiogram showed no endocarditic vegetations. On September, 5th the boy was transferred to the Pediatrics ward for cholecystitis and splenic abscesses. Antimicrobial spectrum was broadened with Claritromicine 250 mg bid per os. Blood tests were performed, including peripheral blood smear, quantitative Ig, total IgE, lymphocyte typing, hemoglobin electrophoresis and dihydrorhodamine 123 test, in order to evaluate any underlying hematologic diseases. On September, 8th, magnetic resonance imaging (MRI) of the abdomen was performed, confirming the presence of seven focal areas of signal alteration, with prevalent peripheral subcapsular distribution (maximum size 1.7 mm).
The morphology of the formations, especially of the most voluminous, was a pyramidal wedge with the apex pointing towards the hilum, and the base of the splenic capsule, while profiles looked finely irregular (<PMC5333739_mjhid-9-1-e2017023f2_undivided_1_1.webp> and 3). The radiographic appearance then made it less likely the possibility of splenic abscesses, in favour of a diagnosis of recent post-infarction outcomes or hematomas in a subacute phase. While all other requested exams resulted negative, on September, 13th, hemoglobin electrophoresis showed a percentage of HbS of 40.6%, HbA was 55.2% while HbA2 and HbF were respectively 3.5 and 0.7%. A diagnosis of SCT was then made.
On September, 15th the boy was discharged, with recommendations to radiologic follow-up and preconception counseling for him and his parents and without further antibiotic therapy.
On October, 13th the boy repeated the US, which showed a normal spleen. The timing of regression is compatible with the diagnosis of splenic infarction."

response:
{"oath": "I will not use the word 'and' in the question unless it is part of a proper noun. I will also make sure the question is concise.", "question": "What hemoglobin electrophoresis result established the diagnosis of sickle cell trait in an 11-year-old boy with splenic lesions after malaria?", "answer": ["While all other requested exams resulted negative, on September, 13th, hemoglobin electrophoresis showed a percentage of HbS of 40.6%, HbA was 55.2% while HbA2 and HbF were respectively 3.5 and 0.7%.", "A diagnosis of SCT was then made."]}
"""

# Use LangChain dedicated template for less headache
CHAT_TEMPLATE = ChatPromptTemplate.from_messages([
    SystemMessage(content=SYS_PROMPT_TEXT),
    ("user", "case_text: {case_text}")
])

**Generate questions and save as .csv file in Google Drive**

In [ ]:
import random

N_GENERATIONS = 10

outputs = []

for sampled_context in tqdm(random.sample(eval_docs, N_GENERATIONS)):
    # Generate QA couple
    case_text = sampled_context.page_content

    output_QA_couple = generate(chat_template=CHAT_TEMPLATE, case_text=case_text)

    # Construct a QA unit pair
    output = {
        "question": output_QA_couple["question"],
        "answer": output_QA_couple["answers"],
        "case_id": sampled_context.metadata["case_id"],
        "case_text": case_text
    }

    outputs.append(output)

In [ ]:
# Process the outputs to join the 'answer' lists into single strings
for item in outputs:
    if 'answer' in item and isinstance(item['answer'], list):
        # Join the list of strings with a space character
        item['answer'] = " ".join(item['answer'])

# Convert QAs into dataframe
QA_df = pd.DataFrame(outputs)

# Drop "oath"
QA_df = QA_df.drop(columns=['oath'], errors='ignore')

PATH = '/content/drive/MyDrive/colab_projects/VGU_WS26_Project/test_generated_qa_pairs.csv'

# Save .csv to Google Drive
QA_df.to_csv(PATH, index=False)

print(f"Successfully saved to {PATH}")